# Step 3 — 取詞建向量空間
以 N-gram 字元切割 + Chi-square 選取鑑別詞，將每篇文章轉為向量，輸出 `vectors.pkl`

Big Data & Business Analytics, National Taiwan University

In [17]:
# ── 可調整參數區 ──────────────────────────────────────
TOP_K_WORDS = 500   # 建構向量空間使用的關鍵字數量（各取一半給看漲/看跌）
# ──────────────────────────────────────────────────────

In [18]:
import csv
import pickle
from collections import Counter
import re


# [METHOD] 目前方法：N-gram 字元切割 + TF-IDF chi-square | 可替換為：jieba/ckip 斷詞、Word2Vec

def get_ngrams(text, min_n=2, max_n=4):
    """對文字做 N-gram 字元切割（不需斷詞套件）"""
    ngrams = []
    for n in range(min_n, max_n + 1):
        for i in range(len(text) - n + 1):
            gram = text[i:i+n]
            if not gram.strip():           # 排除全空白
                continue
            if re.fullmatch(r'[\d\s\W]+', gram):  # 排除純數字/標點/符號
                continue
            if not re.search(r'[\u4e00-\u9fff]', gram):  # 至少含一個中文字
                continue
            ngrams.append(gram)
    return ngrams

In [19]:
# 讀入 articles_labeled.csv，分為看漲 / 看跌 兩組
articles = []
f = open('articles_labeled.csv', newline='', encoding='utf-8')
reader = csv.DictReader(f)
for row in reader:
    row['label'] = int(row['label'])
    articles.append(row)
f.close()

bull_arts = [a for a in articles if a['label'] ==  1]
bear_arts = [a for a in articles if a['label'] == -1]
print(f'看漲文章：{len(bull_arts)} 篇')
print(f'看跌文章：{len(bear_arts)} 篇')

看漲文章：1596 篇
看跌文章：1095 篇


In [12]:
# 分別統計看漲 / 看跌文章的 TF（詞頻）與 DF（文件頻率）
def compute_tf_df(art_list):
    tf = Counter()
    df = Counter()
    for art in art_list:
        text  = (art.get('title', '') or '') + ' ' + (art.get('content', '') or '')
        grams = get_ngrams(text)
        tf.update(grams)          # 累加 TF
        df.update(set(grams))     # 每篇只計一次 DF
    return tf, df

bull_tf, bull_df = compute_tf_df(bull_arts)
bear_tf, bear_df = compute_tf_df(bear_arts)
n_bull = len(bull_arts)
n_bear = len(bear_arts)

print(f'看漲詞彙種類：{len(bull_tf)}')
print(f'看跌詞彙種類：{len(bear_tf)}')

看漲詞彙種類：1002266
看跌詞彙種類：845378


In [13]:
# 計算每個詞的 Chi-square 值（衡量詞與類別的相關性）
# 公式：χ² = N*(A*D - B*C)² / ((A+B)(C+D)(A+C)(B+D))
#   A = 看漲文章含此詞的篇數
#   B = 看跌文章含此詞的篇數
#   C = 看漲文章不含此詞的篇數
#   D = 看跌文章不含此詞的篇數

def chi2_score(term):
    A = bull_df.get(term, 0)
    B = bear_df.get(term, 0)
    C = n_bull - A
    D = n_bear - B
    N = n_bull + n_bear
    denom = (A + B) * (C + D) * (A + C) * (B + D)
    if denom == 0:
        return 0.0
    return N * (A * D - B * C) ** 2 / denom

# 改成：至少在 5 篇以上文章出現
MIN_DF = 5
all_terms = {
    t for t in set(bull_df.keys()) | set(bear_df.keys())
    if bull_df.get(t, 0) + bear_df.get(t, 0) >= MIN_DF
}
print(f'全部候選詞彙：{len(all_terms)} 個，計算 Chi-square...')

chi2 = {term: chi2_score(term) for term in all_terms}
print('完成')

全部候選詞彙：140588 個，計算 Chi-square...
完成


In [20]:
# 計算每個詞的鑑別分數：TF × Chi-square
# 依類別分別排序，各取 TOP_K_WORDS/2 個鑑別詞
half_k = TOP_K_WORDS // 2

bull_score = {t: bull_tf.get(t, 0) * chi2[t] for t in all_terms}
bear_score = {t: bear_tf.get(t, 0) * chi2[t] for t in all_terms}

# 看漲鑑別詞：DF 在看漲文章中較多的詞
bull_candidates = sorted(
    [t for t in all_terms if bull_df.get(t,0) > bear_df.get(t,0)],
    key=lambda t: bull_score[t], reverse=True
)
# 看跌鑑別詞：DF 在看跌文章中較多的詞
bear_candidates = sorted(
    [t for t in all_terms if bear_df.get(t,0) > bull_df.get(t,0)],
    key=lambda t: bear_score[t], reverse=True
)

bull_top = bull_candidates[:half_k]
bear_top = bear_candidates[:half_k]

# 合併為向量空間特徵列表（去重）
seen = set()
feature_list = []
for t in bull_top + bear_top:
    if t not in seen:
        feature_list.append(t)
        seen.add(t)

print(f'特徵詞數量：{len(feature_list)}')
print(f'看漲代表詞（前10）：{bull_top[:10]}')
print(f'看跌代表詞（前10）：{bear_top[:10]}')

特徵詞數量：500
看漲代表詞（前10）：['華城', '緯創', '買超', '士電', '公司', '賣超', '目前', '投信', '營收', '世芯']
看跌代表詞（前10）：['京華', '京華城', '川普', '雲豹', '扣押', '抗告', '機器人', '器人', '鼎越', '越開發']


In [21]:
import math

# 1. 先算每個特徵詞的 IDF
N_docs = len(train_arts)
idf = {}
for term in feature_list:
    df = bull_df.get(term, 0) + bear_df.get(term, 0)
    idf[term] = math.log((N_docs + 1) / (df + 1))

# 2. 向量化改成 TF-IDF（把原本的 vectorize 函數換掉）
def vectorize(text, feature_list):
    """以 TF-IDF 將文章轉為特徵向量"""
    grams      = get_ngrams(text)
    gram_count = Counter(grams)
    return [gram_count.get(f, 0) * idf.get(f, 0) for f in feature_list]

# 3. 這段不用動
train_arts = [a for a in articles if a['label'] != 0]
X = []
y = []
for art in train_arts:
    text = (art.get('title', '') or '') + ' ' + (art.get('content', '') or '')
    X.append(vectorize(text, feature_list))
    y.append(art['label'])

print(f'向量化完成：{len(X)} 篇文章 × {len(feature_list)} 維特徵')

向量化完成：2691 篇文章 × 500 維特徵


In [22]:
# 儲存至 vectors.pkl
f = open('vectors.pkl', 'wb')
pickle.dump({
    'feature_list': feature_list,   # 特徵詞列表
    'X':            X,              # 向量矩陣（list of list）
    'y':            y,              # 標籤列表（1 or -1）
    'articles':     train_arts      # 保留文章 metadata，供時序切割用
}, f)
f.close()

print('已儲存至 vectors.pkl')

已儲存至 vectors.pkl
